<a href="https://colab.research.google.com/github/yanyuelena11/memoryagentbench-recencyretrieval/blob/main/DeepSeekMemoryAgentBench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environment Setup


In [ ]:
!git clone https://github.com/HUST-AI-HYZ/MemoryAgentBench.git
%cd MemoryAgentBench


Cloning into 'MemoryAgentBench'...
remote: Enumerating objects: 1457, done.
remote: Counting objects: 100% (253/253), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 1457 (delta 151), reused 123 (delta 104), pack-reused 1204 (from 1)
Receiving objects: 100% (1457/1457), 2.78 MiB | 12.37 MiB/s, done.
Resolving deltas: 100% (259/259), done.
/content/MemoryAgentBench


In [ ]:
!pip install -q \
  openai tiktoken transformers datasets \
  langchain langchain-openai langchain-community \
  faiss-cpu python-dotenv pyyaml tqdm dill \
  semantic-text-splitter rank-bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q requests==2.32.5


In [ ]:
!pip install -q rouge-score

from rouge_score import rouge_scorer
print("rouge_score installed successfully")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
rouge_score installed successfully


In [ ]:
import nltk
import requests
import openai
import langchain_community
import faiss

nltk.download("punkt")
nltk.download("punkt_tab")

print("requests version:", requests.__version__)
print("Required packages loaded successfully")

requests version: 2.32.5
Required packages loaded successfully


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
%cd /content/MemoryAgentBench

/content/MemoryAgentBench


## 2. API



In [ ]:
from google.colab import userdata
import os

deepseek_key = userdata.get("DEEPSEEK_API_KEY")

# The repository uses the OpenAI-compatible Python library,
# so point that library to DeepSeek.
os.environ["OPENAI_API_KEY"] = deepseek_key
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com"

print("DeepSeek key loaded:", bool(deepseek_key))

DeepSeek key loaded: True


## 3. Compatibility Adjustments

In [ ]:
%cd /content/MemoryAgentBench

!sed -i 's/use_azure=True/use_azure=False/' agent.py

!grep -n "rag_system = RAGSystem" agent.py

/content/MemoryAgentBench
956:        rag_system = RAGSystem(self.retriever, self.model, self.temperature, self.max_tokens, use_azure=False, azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"), azure_api_key=os.environ.get("AZURE_OPENAI_API_KEY"), azure_api_version=os.environ.get("AZURE_OPENAI_API_VERSION"))


In [ ]:
from pathlib import Path
import re

path = Path("/content/MemoryAgentBench/methods/embedding_retriever.py")
text = path.read_text()

pattern = r"max_tokens=self\.max_tokens\s*\n(\s*)\)"
replacement = (
    'max_tokens=self.max_tokens,\n'
    '                                extra_body={"thinking": {"type": "disabled"}}\n'
    r'\1)'
)

if '"type": "disabled"' in text:
    print("Thinking mode was already disabled")
else:
    text, changes = re.subn(pattern, replacement, text, count=1)

    if changes != 1:
        raise RuntimeError("Could not locate the DeepSeek API request")

    path.write_text(text)
    print("Thinking mode disabled successfully")

Thinking mode disabled successfully


## 4. Baseline Configuration

In [ ]:
%%writefile /content/MemoryAgentBench/configs/agent_conf/RAG_Agents/deepseek_contriever.yaml
agent_name: Embedding_rag_contriever
model: deepseek-flash
temperature: 0.0
input_length_limit: 300000
buffer_length: 15000
output_dir: ./outputs/baseline-contriever-512
retrieve_num: 10

Writing /content/MemoryAgentBench/configs/agent_conf/RAG_Agents/deepseek_contriever.yaml


In [ ]:
%cd /content/MemoryAgentBench

!grep -n "from langchain_core.documents" methods/embedding_retriever.py
!grep -n -A2 "max_tokens=self.max_tokens" methods/embedding_retriever.py
!grep -n "use_azure=False" agent.py
!cat configs/agent_conf/RAG_Agents/deepseek_contriever.yaml

/content/MemoryAgentBench
16:from langchain_core.documents import Document
243:                                max_tokens=self.max_tokens,
244-                                extra_body={"thinking": {"type": "disabled"}}
245-                            )
956:        rag_system = RAGSystem(self.retriever, self.model, self.temperature, self.max_tokens, use_azure=False, azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"), azure_api_key=os.environ.get("AZURE_OPENAI_API_KEY"), azure_api_version=os.environ.get("AZURE_OPENAI_API_VERSION"))
agent_name: Embedding_rag_contriever
model: deepseek-flash
temperature: 0.0
input_length_limit: 300000
buffer_length: 15000
output_dir: ./outputs/baseline-contriever-512
retrieve_num: 10


## 5. Testing (5 questions)

In [ ]:
%cd /content/MemoryAgentBench

!python main.py \
  --agent_config configs/agent_conf/RAG_Agents/deepseek_contriever.yaml \
  --dataset_config configs/data_conf/Conflict_Resolution/Factconsolidation_sh_32k.yaml \
  --chunk_size_ablation 512 \
  --max_test_queries_ablation 5

/content/MemoryAgentBench


Using new chunk_size: 512




Using max_test_queries: 5


11/14/2026 09:57:21 - INFO - conversation_creator - Running test on factconsolidation_sh_32k
Dataset: factconsolidation_sh_32k
Loading factconsolidation_sh_32k from Hugging Face dataset: ai-hyz/MemoryAgentBench
11/14/2026 09:57:21 - WARNING - huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
README.md: 100% 9.88k/9.88k [00:00<00:00, 15.2MB/s]

data/Accurate_Retrieval-00000-of-00001.p(…): downloading bytes:  87% 17.4M/20.0M [00:00<00:00, 29.1MB/s, 1.29MB/s  ]
data/Accurate_Retrieval-00000-of-00001.p(…): downloading bytes: 100% 19.6M/19.6M [00:00<00:00, 22.0MB/s, 1.89MB/s  ]
data/Accurate_Retrieval-00000-of-00001.p(…): reconstructing file: 100% 20.0M/20.0M [00:00<00:00, 22.5MB/s, 1.95MB/s  ]
(…)est_Time_Learning-00000-of-00001.parquet: 100% 3.95M/3.95M [00:00<00:00, 49.7MB/s]

data/Long_

In [ ]:
from pathlib import Path
import json

result_path = Path(
    "/content/MemoryAgentBench/outputs/"
    "baseline-contriever-512/Conflict_Resolution/"
    "factconsolidation_sh_32k_unknown_in32768_size10_"
    "shots0_max_samples1_k10_chunk512_results.json"
)

with result_path.open() as file:
    results = json.load(file)

for item in results["data"]:
    print("Question:", item["query"])
    print("Prediction:", item["output"])
    print("Correct answer:", item["answer"])
    print("Exact match:", item["exact_match"])
    print("-" * 60)

Question: Pretend you are a knowledge management system. Each fact in the knowledge pool is provided with a serial number at the beginning, and the newer fact has larger serial number. 
 You need to solve the conflicts of facts in the knowledge pool by finding the newest fact with larger serial number. You need to answer a question based on this rule. You should give a very concise answer without saying other words for the question **only** from the knowledge pool you have memorized rather than the real facts in real world. 

For example:

 [Knowledge Pool] 

 Question: Based on the provided Knowledge Pool, what is the name of the current president of Russia? 
Answer: Donald Trump 

 Now Answer the Question: Based on the provided Knowledge Pool, Which sport is Salem-Keizer Volcanoes associated with? 
Answer:
Prediction: association football
Correct answer: ['basketball']
Exact match: False
------------------------------------------------------------
Question: Pretend you are a knowledg

## 6. Baseline (100 questions)

In [ ]:
%cd /content/MemoryAgentBench

!python main.py \
  --agent_config configs/agent_conf/RAG_Agents/deepseek_contriever.yaml \
  --dataset_config configs/data_conf/Conflict_Resolution/Factconsolidation_sh_32k.yaml \
  --chunk_size_ablation 512

/content/MemoryAgentBench


Using new chunk_size: 512


11/14/2026 10:04:18 - INFO - conversation_creator - Running test on factconsolidation_sh_32k
Dataset: factconsolidation_sh_32k
Loading factconsolidation_sh_32k from Hugging Face dataset: ai-hyz/MemoryAgentBench
11/14/2026 10:04:18 - WARNING - huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loaded 8 samples from Conflict_Resolution
Filtered to 1 samples matching source 'factconsolidation_sh_32k' (from 8 total)
Dataset length: 1, each sample has 100 qa pairs
  0% 0/1 [00:00<?, ?it/s]

 Agent Memorizing...



Processing experiments 1/1: 100% 65/65 [00:00<00:00, 59344.75it/s]


 Agent not saved (not implemented for this agent type) 



!!!!!Processing 100 queries for context 0!!!!!


  0% 0/100 [00:00<?, ?it/s]11/14/2026 10:04:19 - INFO - __main__ - !!!!!Query 0 already processed, skipping...

11/14/2026 10:04:19 - I

In [ ]:
from pathlib import Path
import json

output_directory = Path(
    "/content/MemoryAgentBench/outputs/"
    "baseline-contriever-512/Conflict_Resolution"
)

result_path = next(output_directory.glob("*chunk512_results.json"))

with result_path.open() as file:
    results = json.load(file)

metrics = results["averaged_metrics"]

print("Questions completed:", len(results["data"]))
print("Exact match:", metrics["exact_match"])
print("F1:", metrics["f1"])
print("Substring exact match:", metrics["substring_exact_match"])
print("ROUGE-L F1:", metrics["rougeL_f1"])
print("Average input length:", metrics["input_len"])
print("Average output length:", metrics["output_len"])
print("Average retrieval time:", metrics["memory_construction_time"])
print("Average query time:", metrics["query_time_len"])
print("Result file:", result_path)

Questions completed: 100
Exact match: 27.0
F1: 28.566666666666666
Substring exact match: 28.000000000000004
ROUGE-L F1: 28.566666666666666
Average input length: 6036.65
Average output length: 4.8
Average retrieval time: 0.10331151008605957
Average query time: 1.1659862971305848
Result file: /content/MemoryAgentBench/outputs/baseline-contriever-512/Conflict_Resolution/factconsolidation_sh_32k_unknown_in32768_size10_shots0_max_samples1_k10_chunk512_results.json


In [ ]:
from google.colab import files
files.download(str(result_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import json

result_directory = Path(
    "/content/MemoryAgentBench/outputs/"
    "baseline-contriever-512/Conflict_Resolution"
)

result_files = list(result_directory.glob("*chunk512_results.json"))

if not result_files:
    raise FileNotFoundError("The baseline result JSON was not found.")

result_path = result_files[0]

with result_path.open() as file:
    results = json.load(file)

metrics = results["averaged_metrics"]

number_of_questions = len(results["data"])
exact_match = metrics["exact_match"]
f1 = metrics["f1"]
substring_em = metrics["substring_exact_match"]
rouge_l_f1 = metrics["rougeL_f1"]
average_query_time = metrics["query_time_len"]

print("Number of questions:", number_of_questions)
print(f"Exact match: {exact_match:.2f}%")
print(f"F1: {f1:.2f}%")
print(f"Substring exact match: {substring_em:.2f}%")
print(f"ROUGE-L F1: {rouge_l_f1:.2f}%")
print(f"Average query time: {average_query_time:.2f} seconds")

Number of questions: 100
Exact match: 27.00%
F1: 28.57%
Substring exact match: 28.00%
ROUGE-L F1: 28.57%
Average query time: 1.17 seconds
